In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "mvp_engenharia_de_dados_puc_rio"
SCHEMA = "dados_tse"

TABELA_SILVER = (
    f"{CATALOG}.{SCHEMA}."
    "silver_candidatos_deputado_estadual_sp"
)

silver = spark.table(TABELA_SILVER)

print("========== CAMADA GOLD ==========")
print("Registros Silver:", silver.count())
print("Colunas Silver:", len(silver.columns))

In [0]:
# ============================================================
# GOLD — Etapa 1: validar CPF para comparação entre eleições
# ============================================================

silver_cpf_valido = (
    silver
    .filter(F.col("cpf_valido_para_cruzamento") == True)
)

print("========== CPF VÁLIDO ==========")

for ano in [2022, 2026]:
    df_ano = silver_cpf_valido.filter(
        F.col("ano_eleicao") == ano
    )

    print(f"\n{ano}")
    print("Registros:", df_ano.count())

    print(
        "CPFs distintos:",
        df_ano.select("cpf_candidato").distinct().count()
    )

    print(
        "Registros excedentes por CPF duplicado:",
        df_ano.count()
        - df_ano.select("cpf_candidato").distinct().count()
    )

In [0]:
# ============================================================
# GOLD — Etapa 2: analisar CPFs válidos duplicados em 2022
# ============================================================

cpfs_duplicados_2022 = (
    silver_cpf_valido
    .filter(F.col("ano_eleicao") == 2022)
    .groupBy("cpf_candidato")
    .count()
    .filter(F.col("count") > 1)
    .select("cpf_candidato")
)

duplicados_2022 = (
    silver_cpf_valido
    .filter(F.col("ano_eleicao") == 2022)
    .join(
        cpfs_duplicados_2022,
        on="cpf_candidato",
        how="inner"
    )
)

print(
    "Quantidade de CPFs válidos duplicados:",
    cpfs_duplicados_2022.count()
)

display(
    duplicados_2022
    .select(
        "cpf_candidato",
        "sq_candidato",
        "nome_candidato",
        "nr_candidato",
        "sigla_partido",
        "genero",
        "cor_raca",
        "situacao_candidatura",
        "detalhe_situacao_candidatura"
    )
    .orderBy(
        "cpf_candidato",
        "sq_candidato"
    )
)

In [0]:
# ============================================================
# GOLD — Etapa 3:
# consolidar uma linha por CPF válido em cada eleição
# ============================================================

janela_cpf_ano = (
    Window
    .partitionBy("ano_eleicao", "cpf_candidato")
    .orderBy(
        F.when(
            F.col("situacao_candidatura") == "APTO", 1
        ).otherwise(2),
        F.when(
            F.col("detalhe_situacao_candidatura") == "DEFERIDO", 1
        ).otherwise(2),
        F.col("sq_candidato").desc()
    )
)

silver_pessoa_ano = (
    silver_cpf_valido
    .withColumn(
        "ordem_registro_cpf",
        F.row_number().over(janela_cpf_ano)
    )
    .filter(F.col("ordem_registro_cpf") == 1)
    .drop("ordem_registro_cpf")
)

print("========== CONTROLE PESSOA / ANO ==========")

for ano in [2022, 2026]:
    df_ano = silver_pessoa_ano.filter(
        F.col("ano_eleicao") == ano
    )

    print(f"\n{ano}")
    print("Registros:", df_ano.count())
    print(
        "CPFs distintos:",
        df_ano.select("cpf_candidato").distinct().count()
    )

In [0]:
# ============================================================
# GOLD — Etapa 4:
# comparação de candidatos entre 2022 e 2026
# ============================================================

pessoas_2022 = (
    silver_pessoa_ano
    .filter(F.col("ano_eleicao") == 2022)
    .select(
        "cpf_candidato",
        F.col("nome_candidato").alias("nome_2022"),
        F.col("sq_candidato").alias("sq_candidato_2022"),
        F.col("nr_candidato").alias("nr_candidato_2022"),
        F.col("sigla_partido").alias("partido_2022"),
        F.col("genero").alias("genero_2022"),
        F.col("cor_raca").alias("cor_raca_2022"),
        F.col("situacao_candidatura").alias("situacao_2022")
    )
)

pessoas_2026 = (
    silver_pessoa_ano
    .filter(F.col("ano_eleicao") == 2026)
    .select(
        "cpf_candidato",
        F.col("nome_candidato").alias("nome_2026"),
        F.col("sq_candidato").alias("sq_candidato_2026"),
        F.col("nr_candidato").alias("nr_candidato_2026"),
        F.col("sigla_partido").alias("partido_2026"),
        F.col("genero").alias("genero_2026"),
        F.col("cor_raca").alias("cor_raca_2026"),
        F.col("situacao_candidatura").alias("situacao_2026")
    )
)

gold_comparacao = (
    pessoas_2022
    .join(
        pessoas_2026,
        on="cpf_candidato",
        how="full"
    )
    .withColumn(
        "participou_2022",
        F.col("sq_candidato_2022").isNotNull()
    )
    .withColumn(
        "participou_2026",
        F.col("sq_candidato_2026").isNotNull()
    )
    .withColumn(
        "situacao_comparacao",
        F.when(
            F.col("participou_2022") & F.col("participou_2026"),
            "AMBOS_ANOS"
        )
        .when(
            F.col("participou_2022"),
            "SOMENTE_2022"
        )
        .otherwise("SOMENTE_2026")
    )
    .withColumn(
        "mudou_partido",
        F.when(
            F.col("participou_2022") &
            F.col("participou_2026"),
            F.col("partido_2022") != F.col("partido_2026")
        )
    )
    .withColumn(
        "nome_candidato",
        F.coalesce(
            F.col("nome_2026"),
            F.col("nome_2022")
        )
    )
)

print("========== CONTROLE GOLD COMPARAÇÃO ==========")
print("Registros:", gold_comparacao.count())

print("\nSituação de participação:")

(
    gold_comparacao
    .groupBy("situacao_comparacao")
    .count()
    .orderBy("situacao_comparacao")
    .show()
)

print("\nMudança de partido entre candidatos presentes nos dois anos:")

(
    gold_comparacao
    .filter(F.col("situacao_comparacao") == "AMBOS_ANOS")
    .groupBy("mudou_partido")
    .count()
    .show()
)

In [0]:
# ============================================================
# GOLD — Etapa 5: gravar tabela de comparação 2022 x 2026
# ============================================================

TABELA_GOLD_COMPARACAO = (
    "mvp_engenharia_de_dados_puc_rio.dados_tse."
    "gold_comparacao_candidatos_2022_2026"
)

gold_comparacao_final = (
    gold_comparacao
    .select(
        "cpf_candidato",
        "nome_candidato",

        "participou_2022",
        "participou_2026",
        "situacao_comparacao",

        "nome_2022",
        "sq_candidato_2022",
        "nr_candidato_2022",
        "partido_2022",
        "genero_2022",
        "cor_raca_2022",
        "situacao_2022",

        "nome_2026",
        "sq_candidato_2026",
        "nr_candidato_2026",
        "partido_2026",
        "genero_2026",
        "cor_raca_2026",
        "situacao_2026",

        "mudou_partido"
    )
)

(
    gold_comparacao_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_GOLD_COMPARACAO)
)

print(
    "Tabela Gold criada com sucesso:",
    TABELA_GOLD_COMPARACAO
)

In [0]:
gold_comparacao_gravada = spark.table(
    TABELA_GOLD_COMPARACAO
)

print("========== CONTROLE GOLD GRAVADA ==========")

print(
    "Registros:",
    gold_comparacao_gravada.count()
)

print(
    "CPFs distintos:",
    gold_comparacao_gravada
    .select("cpf_candidato")
    .distinct()
    .count()
)

print("\nSituação de comparação:")

(
    gold_comparacao_gravada
    .groupBy("situacao_comparacao")
    .count()
    .orderBy("situacao_comparacao")
    .show()
)

print("\nMudança de partido — ambos os anos:")

(
    gold_comparacao_gravada
    .filter(
        F.col("situacao_comparacao") == "AMBOS_ANOS"
    )
    .groupBy("mudou_partido")
    .count()
    .show()
)

In [0]:
# ============================================================
# GOLD — Etapa 6: distribuição por cor/raça
# ============================================================

# Quantidade por ano e categoria
raca_base = (
    silver
    .groupBy("ano_eleicao", "cor_raca")
    .agg(
        F.count("*").alias("quantidade")
    )
)

# Total de candidaturas de cada ano
janela_ano = Window.partitionBy("ano_eleicao")

raca_base = (
    raca_base
    .withColumn(
        "total_ano",
        F.sum("quantidade").over(janela_ano)
    )
    .withColumn(
        "percentual",
        F.round(
            F.col("quantidade") /
            F.col("total_ano") * 100,
            2
        )
    )
)

# Separar os anos para comparação
raca_2022 = (
    raca_base
    .filter(F.col("ano_eleicao") == 2022)
    .select(
        "cor_raca",
        F.col("quantidade").alias("qtd_2022"),
        F.col("percentual").alias("pct_2022")
    )
)

raca_2026 = (
    raca_base
    .filter(F.col("ano_eleicao") == 2026)
    .select(
        "cor_raca",
        F.col("quantidade").alias("qtd_2026"),
        F.col("percentual").alias("pct_2026")
    )
)

# Comparação 2022 x 2026
gold_raca = (
    raca_2022
    .join(
        raca_2026,
        on="cor_raca",
        how="full"
    )
    .fillna(
        0,
        subset=[
            "qtd_2022",
            "pct_2022",
            "qtd_2026",
            "pct_2026"
        ]
    )
    .withColumn(
        "variacao_qtd_pct",
        F.when(
            F.col("qtd_2022") > 0,
            F.round(
                (
                    F.col("qtd_2026") -
                    F.col("qtd_2022")
                ) /
                F.col("qtd_2022") * 100,
                2
            )
        )
    )
    .withColumn(
        "variacao_pontos_percentuais",
        F.round(
            F.col("pct_2026") -
            F.col("pct_2022"),
            2
        )
    )
    .orderBy("cor_raca")
)

display(gold_raca)

In [0]:
# ============================================================
# GOLD — Etapa 7: gravar distribuição de raça
# ============================================================

TABELA_GOLD_RACA = (
    "mvp_engenharia_de_dados_puc_rio.dados_tse."
    "gold_distribuicao_raca"
)

(
    gold_raca.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_GOLD_RACA)
)

print(
    "Tabela Gold criada com sucesso:",
    TABELA_GOLD_RACA
)

In [0]:
# ============================================================
# GOLD — Etapa 8: distribuição por gênero
# ============================================================

genero_base = (
    silver
    .groupBy("ano_eleicao", "genero")
    .agg(
        F.count("*").alias("quantidade")
    )
)

janela_ano = Window.partitionBy("ano_eleicao")

genero_base = (
    genero_base
    .withColumn(
        "total_ano",
        F.sum("quantidade").over(janela_ano)
    )
    .withColumn(
        "percentual",
        F.round(
            F.col("quantidade") /
            F.col("total_ano") * 100,
            2
        )
    )
)

genero_2022 = (
    genero_base
    .filter(F.col("ano_eleicao") == 2022)
    .select(
        "genero",
        F.col("quantidade").alias("qtd_2022"),
        F.col("percentual").alias("pct_2022")
    )
)

genero_2026 = (
    genero_base
    .filter(F.col("ano_eleicao") == 2026)
    .select(
        "genero",
        F.col("quantidade").alias("qtd_2026"),
        F.col("percentual").alias("pct_2026")
    )
)

gold_genero = (
    genero_2022
    .join(
        genero_2026,
        on="genero",
        how="full"
    )
    .fillna(
        0,
        subset=[
            "qtd_2022",
            "pct_2022",
            "qtd_2026",
            "pct_2026"
        ]
    )
    .withColumn(
        "variacao_qtd_pct",
        F.when(
            F.col("qtd_2022") > 0,
            F.round(
                (
                    F.col("qtd_2026") -
                    F.col("qtd_2022")
                ) /
                F.col("qtd_2022") * 100,
                2
            )
        )
    )
    .withColumn(
        "variacao_pontos_percentuais",
        F.round(
            F.col("pct_2026") -
            F.col("pct_2022"),
            2
        )
    )
    .orderBy("genero")
)

display(gold_genero)

In [0]:
# ============================================================
# GOLD — Etapa 9: gravar distribuição de gênero
# ============================================================

TABELA_GOLD_GENERO = (
    "mvp_engenharia_de_dados_puc_rio.dados_tse."
    "gold_distribuicao_genero"
)

(
    gold_genero.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_GOLD_GENERO)
)

print(
    "Tabela Gold criada com sucesso:",
    TABELA_GOLD_GENERO
)

In [0]:
# ============================================================
# GOLD — Etapa 10: controle final da camada Gold
# ============================================================

tabelas_gold = [
    "gold_comparacao_candidatos_2022_2026",
    "gold_distribuicao_raca",
    "gold_distribuicao_genero"
]

print("========== CONTROLE FINAL DA CAMADA GOLD ==========")

for tabela in tabelas_gold:
    nome_completo = (
        f"mvp_engenharia_de_dados_puc_rio."
        f"dados_tse.{tabela}"
    )

    df = spark.table(nome_completo)

    print(
        f"{tabela}: "
        f"{df.count()} registros | "
        f"{len(df.columns)} colunas"
    )

print("\n========== CONTROLE DA COMPARAÇÃO ==========")

gold_cmp = spark.table(
    "mvp_engenharia_de_dados_puc_rio.dados_tse."
    "gold_comparacao_candidatos_2022_2026"
)

(
    gold_cmp
    .groupBy("situacao_comparacao")
    .count()
    .orderBy("situacao_comparacao")
    .show()
)

print("\n========== CONTROLE DE MUDANÇA DE PARTIDO ==========")

(
    gold_cmp
    .filter(F.col("situacao_comparacao") == "AMBOS_ANOS")
    .groupBy("mudou_partido")
    .count()
    .show()
)